# Chapter 11 · Probability: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only; they have no correlation with the real world.

**Part A** follows the chapter's case, the delivery register of a public maternity unit, through the committee's six questions. **Part B** gives the written solution of every chapter exercise (statements are in the book) and, for the exercises whose solution needs a calculation, its code and result. Run the cells in order.

## Part A. The case

**Data.** `data/deliveries.csv`: 5,000 rows, one per delivery, and four columns: `delivery_id`, `cesarean`, `hemorrhage`, `infection` (yes/no). Each cell carries the number of the book's section (11.1, 11.2, ...), so you can follow the chapter and the notebook side by side.

### 11.1 Introduction: the data

In [1]:
import pandas as pd

df = pd.read_csv("data/deliveries.csv")

# The size of the register: rows (deliveries) and columns
rows, columns = df.shape
print("The register has", rows, "rows (one per delivery) and", columns, "columns.")
df.head()                          # the first five rows

The register has 5000 rows (one per delivery) and 4 columns.


,delivery_id,cesarean,hemorrhage,infection
0,D0001,no,no,no
1,D0002,no,no,no
2,D0003,yes,no,no
3,D0004,no,no,no
4,D0005,yes,no,no


### 11.2 Fractions in the register

*What fraction of deliveries were cesarean, and what fraction had a hemorrhage?*

In [2]:
# 1. Take all the rows: one row is one delivery
n = len(df)
print("The register has", n, "deliveries.")

# 2. In each row, look at the column: is it "yes"?
C = df["cesarean"] == "yes"        # True for each cesarean delivery
H = df["hemorrhage"] == "yes"
I = df["infection"] == "yes"
n_cesarean = C.sum()
print("Of the", n, "deliveries,", n_cesarean, "were cesarean and", n - n_cesarean, "were not.")

# 3. Count the "yes" rows and divide by all the rows
n_hemorrhage = H.sum()
print("Cesarean:", n_cesarean, "of", n, "deliveries =", round(100 * n_cesarean / n, 2), "%")
print("Hemorrhage:", n_hemorrhage, "of", n, "deliveries =", round(100 * n_hemorrhage / n, 2), "%")

The register has 5000 deliveries.
Of the 5000 deliveries, 1313 were cesarean and 3687 were not.
Cesarean: 1313 of 5000 deliveries = 26.26 %
Hemorrhage: 290 of 5000 deliveries = 5.8 %


### 11.2.1 Experiment, sample space, and event

In [3]:
# The sample space is the 5,000 records; the events H and C are the records with hemorrhage and with cesarean
# With every record equally likely: P(event) = records in the event / all records
p_hemorrhage = n_hemorrhage / n
p_cesarean = n_cesarean / n
print("P(H) =", round(p_hemorrhage, 4), "and P(C) =", round(p_cesarean, 4))
print("Chance that the next mother admitted has a hemorrhage:", round(100 * p_hemorrhage, 2), "%")

P(H) = 0.058 and P(C) = 0.2626
Chance that the next mother admitted has a hemorrhage: 5.8 %


### 11.2.2 Why are many cases needed in a data analysis?

In [4]:
# Read the register in order: does the fraction of cesareans settle?
for k in [10, 50, 100, 500, 1000, 5000]:
    first_k = C.iloc[:k]           # the first k rows
    print("After", k, "rows:", first_k.sum(), "cesareans =", round(100 * first_k.mean(), 2), "%")

After 10 rows: 5 cesareans = 50.0 %
After 50 rows: 19 cesareans = 38.0 %
After 100 rows: 27 cesareans = 27.0 %
After 500 rows: 123 cesareans = 24.6 %
After 1000 rows: 246 cesareans = 24.6 %
After 5000 rows: 1313 cesareans = 26.26 %


### 11.3 Hemorrhage or infection: count each mother once

*What is the chance that the next mother admitted has a hemorrhage or an infection?*

In [5]:
# 1. Cross the two columns: each delivery falls in exactly one of four cells
table = pd.crosstab(df["hemorrhage"], df["infection"], margins=True, margins_name="Total")
table.index = ["No hemorrhage", "Hemorrhage", "Total"]      # rows
table.columns = ["No infection", "Infection", "Total"]      # columns
display(table)                                             # shows the table formatted

# 2. Count the mothers with each complication, and with both
n_infection = I.sum()
n_both = (H & I).sum()             # "&" means "and": both at the same time
print()
print(n_hemorrhage, "mothers had a hemorrhage and", n_infection, "had an infection.")
print("Adding the two gives", n_hemorrhage + n_infection, "but the", n_both, "mothers with both were counted twice.")

,No infection,Infection,Total
No hemorrhage,4599,111,4710
Hemorrhage,271,19,290
Total,4870,130,5000



290 mothers had a hemorrhage and 130 had an infection.
Adding the two gives 420 but the 19 mothers with both were counted twice.


### 11.3.1 Rules for combining the counts

In [6]:
# 1. Addition rule: count the mothers with both only once
print("Hemorrhage or infection:", n_hemorrhage, "+", n_infection, "-", n_both, "=", n_hemorrhage + n_infection - n_both, "mothers.")

# 2. Check by counting directly
either = H | I                     # "|" means "or": at least one of the two
n_either = either.sum()
print("Counted directly:", n_either, "of", n, "mothers =", round(100 * n_either / n, 2), "%")

# 3. Complement: the mothers with neither
print("With neither:", n - n_either, "of", n, "mothers =", round(100 * (n - n_either) / n, 2), "%")

Hemorrhage or infection: 290 + 130 - 19 = 401 mothers.
Counted directly: 401 of 5000 mothers = 8.02 %
With neither: 4599 of 5000 mothers = 91.98 %


### 11.4 Infection within each delivery group

*Is infection more frequent after a cesarean?*

In [7]:
# 1. Split the deliveries by type: cesarean, and vaginal (not cesarean)
n_vaginal = (~C).sum()             # "~" means "not"

# 2. Within each group, count the infections and divide by the size of that group
infected_cesarean = (C & I).sum()
infected_vaginal = (~C & I).sum()
print("Infection among cesarean deliveries:", infected_cesarean, "of", n_cesarean, "=", round(infected_cesarean / n_cesarean, 4))
print("Infection among vaginal deliveries:", infected_vaginal, "of", n_vaginal, "=", round(infected_vaginal / n_vaginal, 4))

Infection among cesarean deliveries: 87 of 1313 = 0.0663
Infection among vaginal deliveries: 43 of 3687 = 0.0117


### 11.4.1 The same count can answer different questions

In [8]:
# 1. A mother admitted here has a vaginal delivery and, after that, an infection: two steps
p_vaginal = n_vaginal / n                                  # step 1: vaginal delivery
p_infection_after_vaginal = infected_vaginal / n_vaginal   # step 2: infection, among vaginal deliveries
print("Vaginal delivery:", round(100 * p_vaginal, 2), "% of the mothers; infection among them:", round(100 * p_infection_after_vaginal, 2), "%")
print("Vaginal delivery and then infection:", round(p_vaginal, 4), "x", round(p_infection_after_vaginal, 4), "=", round(p_vaginal * p_infection_after_vaginal, 4), "(", infected_vaginal, "of", n, ")")

# 2. Another mother has a cesarean and, after that, an infection: the two steps change
p_cesarean = n_cesarean / n                                # step 1: cesarean
p_infection_after_cesarean = infected_cesarean / n_cesarean  # step 2: infection, among cesareans
print()
print("Cesarean:", round(100 * p_cesarean, 2), "% of the mothers; infection among them:", round(100 * p_infection_after_cesarean, 2), "%")
print("Cesarean and then infection:", round(p_cesarean, 4), "x", round(p_infection_after_cesarean, 4), "=", round(p_cesarean * p_infection_after_cesarean, 4), "(", infected_cesarean, "of", n, ")")

# 3. Total probability: the two paths added give the chance of an infection over all deliveries
total_probability = p_vaginal * p_infection_after_vaginal + p_cesarean * p_infection_after_cesarean
print()
print("Total probability of infection:", round(total_probability, 4), "; direct count:", n_infection, "of", n, "=", round(n_infection / n, 4))

Vaginal delivery: 73.74 % of the mothers; infection among them: 1.17 %
Vaginal delivery and then infection: 0.7374 x 0.0117 = 0.0086 ( 43 of 5000 )

Cesarean: 26.26 % of the mothers; infection among them: 6.63 %
Cesarean and then infection: 0.2626 x 0.0663 = 0.0174 ( 87 of 5000 )

Total probability of infection: 0.026 ; direct count: 130 of 5000 = 0.026


### 11.5 Do hemorrhage and infection occur independently?

*Does knowing that the next mother admitted had a hemorrhage change her chance of an infection?*

In [9]:
# 1. An identifier is only a label: is "identifier in the first half of the register" independent of hemorrhage?
number = df["delivery_id"].str[1:].astype(int)       # D0001 -> 1
F = number <= 2500                                    # True for D0001 to D2500
n_first = F.sum()
n_first_hemorrhage = (F & H).sum()
print("Expected if independent:", round(n_first / n, 4), "x", round(n_hemorrhage / n, 4), "=", round(n_first / n * n_hemorrhage / n, 4))
print("Observed, both together:", n_first_hemorrhage, "of", n, "=", round(n_first_hemorrhage / n, 4))
print("Hemorrhage in the first half:", round(n_first_hemorrhage / n_first, 4), "; in the second half:", round((~F & H).sum() / (~F).sum(), 4))

# 2. The contrast: hemorrhage and infection
p_h = n_hemorrhage / n
p_i = n_infection / n
print()
print("Expected if independent:", round(p_h, 4), "x", round(p_i, 4), "=", round(p_h * p_i, 6))
print("Observed, both together:", n_both, "of", n, "=", round(n_both / n, 4))

# 3. Infection within the mothers with and without hemorrhage
print("Infection among mothers with hemorrhage:", n_both, "of", n_hemorrhage, "=", round(n_both / n_hemorrhage, 4))
print("Infection among mothers without hemorrhage:", (~H & I).sum(), "of", (~H).sum(), "=", round((~H & I).sum() / (~H).sum(), 4))

Expected if independent: 0.5 x 0.058 = 0.029
Observed, both together: 145 of 5000 = 0.029
Hemorrhage in the first half: 0.058 ; in the second half: 0.058

Expected if independent: 0.058 x 0.026 = 0.001508
Observed, both together: 19 of 5000 = 0.0038
Infection among mothers with hemorrhage: 19 of 290 = 0.0655
Infection among mothers without hemorrhage: 111 of 4710 = 0.0236


### 11.6 Among infections, how many followed cesareans?

*Among the mothers with infection, what fraction had a cesarean?*

In [10]:
# 1. The table: rows are the delivery type, columns are infection
table = pd.crosstab(df["cesarean"], df["infection"], margins=True, margins_name="Total")
table.index = ["Vaginal", "Cesarean", "Total"]              # rows
table.columns = ["No infection", "Infection", "Total"]      # columns
display(table)                                             # shows the table formatted

# 2. Counting directly among the infected mothers
print()
print("Among the", n_infection, "infections,", infected_cesarean, "followed a cesarean =", round(100 * infected_cesarean / n_infection, 2), "%")

# 3. The same by reversing the condition (Bayes): P(C|I) = P(I|C) P(C) / P(I)
print("Reversal:", round(infected_cesarean / n_cesarean, 4), "x", round(n_cesarean / n, 4), "/", round(total_probability, 4), "=", round((infected_cesarean / n_cesarean) * (n_cesarean / n) / total_probability, 4))

,No infection,Infection,Total
Vaginal,3644,43,3687
Cesarean,1226,87,1313
Total,4870,130,5000



Among the 130 infections, 87 followed a cesarean = 66.92 %
Reversal: 0.0663 x 0.2626 / 0.026 = 0.6692


### 11.7 Choosing charts for review

*In how many ways can 20 charts be chosen, and how likely is the sample to include an infection?*

In [11]:
from math import comb, factorial, perm

total = n                          # charts in the register
without_infection = n - n_infection

# In how many ways can 20 charts be chosen? (order does not matter)
sets = comb(total, 20)
print("There are", sets, "different sets of 20 charts,", len(str(sets)), "digits.")

There are 37735558620382470911680923426535607627307325009655406000 different sets of 20 charts, 56 digits.


### 11.7.1 Does order matter?

In [12]:
# Orders of the 20 selected charts, and of the first two slots
print("The 20 chosen charts can be put in", factorial(20), "orders; the first two slots, in", perm(20, 2))

The 20 chosen charts can be put in 2432902008176640000 orders; the first two slots, in 380


### 11.7.2 Use the sample with no infections as the complement

In [13]:
# 1. Chance of at least one infection = 1 - chance of none (all 20 from the charts without infection)
chance_none = comb(without_infection, 20) / comb(total, 20)
print("Chance that 20 charts include at least one infection:", round(1 - chance_none, 4))

# 2. Smallest sample with at least 90%: try sizes 1, 2, 3, ... until the chance of none is at most 10%
size = 1
while 10 * comb(without_infection, size) > comb(total, size):     # compares whole numbers, no rounding
    size = size + 1
for m in [size - 1, size]:
    print("With", m, "charts the chance is", round(1 - comb(without_infection, m) / comb(total, m), 6))
print("So", size, "is the smallest sample with at least 90%.")

Chance that 20 charts include at least one infection: 0.4102
With 86 charts the chance is 0.898259
With 87 charts the chance is 0.90095
So 87 is the smallest sample with at least 90%.


## Part B. Exercise solutions

### 11.9 Exercise 1. Experiment, sample space, and marginal probabilities

The experiment is choosing one of the 5,000 records at random, with every record equally likely. The sample space is the set of the 5,000 records. $H$, $C$, and $I$ are the records with hemorrhage (290), cesarean delivery (1,313), and infection (130). Then $P(H)=290/5{,}000=0.0580$, $P(C)=1{,}313/5{,}000=0.2626$, and $P(I)=130/5{,}000=0.0260$. Each concerns one event alone, so all three are marginal probabilities.

In [14]:
# 1. The sample space is the 5,000 records; the events H, C, and I are the records with each "yes"
print("Records:", n)
print("P(H) =", n_hemorrhage, "/", n, "=", round(n_hemorrhage / n, 4))
print("P(C) =", n_cesarean, "/", n, "=", round(n_cesarean / n, 4))
print("P(I) =", n_infection, "/", n, "=", round(n_infection / n, 4))

Records: 5000
P(H) = 290 / 5000 = 0.058
P(C) = 1313 / 5000 = 0.2626
P(I) = 130 / 5000 = 0.026


### 11.9 Exercise 2. Why many cases are needed

The fractions of hemorrhage are 0/10 = 0%, 1/50 = 2%, 7/100 = 7%, 25/500 = 5%, 47/1,000 = 4.7%, and 290/5,000 = 5.8%. From 10 to 100 records the fraction changes by 7.0 percentage points; from 1,000 to 5,000 records, by only 1.1. A fraction from few cases can be far from the complete register's, and as records accumulate it approaches 5.80%, as the law of large numbers describes. The gain per added record shrinks, which is the diminishing return: for independent observations the typical error falls with the square root of the number of cases, whereas here the records are read in a fixed order.

In [15]:
# 1. Fraction of hemorrhage among the first k records
for k in [10, 50, 100, 500, 1000, 5000]:
    first_k = H.iloc[:k]
    print("After", k, "rows:", first_k.sum(), "hemorrhages =", round(100 * first_k.mean(), 2), "%")

# 2. Change from 10 to 100 records and from 1,000 to 5,000 records, in percentage points
print()
print("From 10 to 100 records:", round(100 * (H.iloc[:100].mean() - H.iloc[:10].mean()), 2), "points")
print("From 1000 to 5000 records:", round(100 * (H.mean() - H.iloc[:1000].mean()), 2), "points")

After 10 rows: 0 hemorrhages = 0.0 %
After 50 rows: 1 hemorrhages = 2.0 %
After 100 rows: 7 hemorrhages = 7.0 %
After 500 rows: 25 hemorrhages = 5.0 %
After 1000 rows: 47 hemorrhages = 4.7 %
After 5000 rows: 290 hemorrhages = 5.8 %

From 10 to 100 records: 7.0 points
From 1000 to 5000 records: 1.1 points


### 11.9 Exercise 3. OR, AND, NOT, and mutually exclusive events

The four groups are hemorrhage only (271), infection only (111), both (19), and neither (4,599), with probabilities 0.0542, 0.0222, 0.0038, and 0.9198. The addition rule gives $290+130-19=401$ mothers, or 0.0802, and direct counting of “hemorrhage OR infection” also gives 401. Cesarean and vaginal delivery are mutually exclusive: no record is both (0 records), and $P(C\text{ or not }C)=1$. Hemorrhage and infection are not mutually exclusive, because 19 records have both.

In [16]:
# 1. The four groups: hemorrhage only, infection only, both, neither
only_h = (H & ~I).sum()
only_i = (~H & I).sum()
both = (H & I).sum()
neither = (~H & ~I).sum()
print("Hemorrhage only:", only_h, "mothers =", round(only_h / n, 4))
print("Infection only:", only_i, "mothers =", round(only_i / n, 4))
print("Both:", both, "mothers =", round(both / n, 4))
print("Neither:", neither, "mothers =", round(neither / n, 4))

# 2. Addition rule for "hemorrhage or infection", and the direct count
print()
print("Addition rule:", n_hemorrhage + n_infection - both, "mothers =", round((n_hemorrhage + n_infection - both) / n, 4))
print("Direct count:", (H | I).sum(), "mothers")

# 3. Mutually exclusive events cannot occur together: count the records with both
print()
print("Cesarean and vaginal together:", (C & ~C).sum(), "records")
print("Hemorrhage and infection together:", both, "records")

Hemorrhage only: 271 mothers = 0.0542
Infection only: 111 mothers = 0.0222
Both: 19 mothers = 0.0038
Neither: 4599 mothers = 0.9198

Addition rule: 401 mothers = 0.0802
Direct count: 401 mothers

Cesarean and vaginal together: 0 records
Hemorrhage and infection together: 19 records


### 11.9 Exercise 4. Marginal, joint, and conditional fractions

(a) Conditional, $P(I\mid C)=87/1{,}313\approx0.0663$. (b) Joint, $P(C\text{ and }I)=87/5{,}000=0.0174$. (c) Marginal, $P(I)=130/5{,}000=0.0260$. (d) Conditional, $P(H\mid C)=95/1{,}313\approx0.0724$. (e) Joint, $P(\text{not }C\text{ and }I)=43/5{,}000=0.0086$. The denominators differ: the group for a conditional probability, all 5,000 deliveries for the others.

In [17]:
# 1. The five fractions: the numerator and the denominator of each
print("(a) infection among cesareans:", (C & I).sum(), "/", n_cesarean, "=", round((C & I).sum() / n_cesarean, 4))
print("(b) cesarean and infection, among all:", (C & I).sum(), "/", n, "=", round((C & I).sum() / n, 4))
print("(c) infection, among all:", n_infection, "/", n, "=", round(n_infection / n, 4))
print("(d) hemorrhage among cesareans:", (C & H).sum(), "/", n_cesarean, "=", round((C & H).sum() / n_cesarean, 4))
print("(e) vaginal delivery and infection, among all:", (~C & I).sum(), "/", n, "=", round((~C & I).sum() / n, 4))

(a) infection among cesareans: 87 / 1313 = 0.0663
(b) cesarean and infection, among all: 87 / 5000 = 0.0174
(c) infection, among all: 130 / 5000 = 0.026
(d) hemorrhage among cesareans: 95 / 1313 = 0.0724
(e) vaginal delivery and infection, among all: 43 / 5000 = 0.0086


### 11.9 Exercise 5. Two-step chances and total probability

Vaginal delivery then infection: $P(\text{not }C)P(I\mid\text{not }C)\approx0.7374\times0.0117\approx0.0086$, which is $43/5{,}000$. Cesarean then infection: $P(C)P(I\mid C)\approx0.2626\times0.0663\approx0.0174$, which is $87/5{,}000$. Adding them gives $0.0086+0.0174=0.0260=P(I)$, the total probability of infection. The cesarean path is about twice as likely ($0.0174/0.0086\approx2.02$), although its first step is much smaller, because infection after a cesarean is much more frequent.

In [18]:
# 1. Vaginal delivery, then infection; cesarean, then infection: first step times second step
n_vaginal = (~C).sum()
path_vaginal = (n_vaginal / n) * ((~C & I).sum() / n_vaginal)
path_cesarean = (n_cesarean / n) * ((C & I).sum() / n_cesarean)
print("Vaginal delivery, then infection:", round(path_vaginal, 4), "(direct count:", round((~C & I).sum() / n, 4), ")")
print("Cesarean, then infection:", round(path_cesarean, 4), "(direct count:", round((C & I).sum() / n, 4), ")")

# 2. The two paths added are the total probability of infection, and their ratio
print()
print("Total probability of infection:", round(path_vaginal + path_cesarean, 4), "; direct count:", round(n_infection / n, 4))
print("Cesarean path divided by vaginal path:", round(path_cesarean / path_vaginal, 2))

Vaginal delivery, then infection: 0.0086 (direct count: 0.0086 )
Cesarean, then infection: 0.0174 (direct count: 0.0174 )

Total probability of infection: 0.026 ; direct count: 0.026
Cesarean path divided by vaginal path: 2.02


### 11.9 Exercise 6. Independence

(a) $P(O)P(C)=0.5\times0.2626=0.1313$ and $P(O\text{ and }C)=667/5{,}000=0.1334$; $P(C\mid O)=667/2{,}500=0.2668$ and $P(C\mid\text{not }O)=646/2{,}500=0.2584$. They are not exactly independent in this register, and an exact match is impossible here, because $2{,}500\times0.2626$ is not a whole number. A gap this small is not surprising for an identifier, which is only a label; whether it reflects more than chance is a question for a hypothesis test. (b) $P(C)P(H)=0.2626\times0.0580\approx0.0152$ and $P(C\text{ and }H)=95/5{,}000=0.0190$; $P(H\mid C)\approx0.0724$ and $P(H\mid\text{not }C)\approx0.0529$. This gap is larger, and cesarean and hemorrhage look dependent. The data are synthetic and these are didactic calculations.

In [19]:
# 1. O = the identifier is an odd number: compare P(O and C) with P(O) x P(C)
O = number % 2 == 1
print("O and C:", (O & C).sum(), "of", n, "=", round((O & C).sum() / n, 4), "; P(O) x P(C) =", round(O.mean() * C.mean(), 4))
print("Cesarean among odd identifiers:", round(C[O].mean(), 4), "; among even identifiers:", round(C[~O].mean(), 4))

# 2. Cesarean and hemorrhage
print()
print("C and H:", (C & H).sum(), "of", n, "=", round((C & H).sum() / n, 4), "; P(C) x P(H) =", round(C.mean() * H.mean(), 4))
print("Hemorrhage among cesareans:", round(H[C].mean(), 4), "; among vaginal deliveries:", round(H[~C].mean(), 4))

O and C: 667 of 5000 = 0.1334 ; P(O) x P(C) = 0.1313
Cesarean among odd identifiers: 0.2668 ; among even identifiers: 0.2584

C and H: 95 of 5000 = 0.019 ; P(C) x P(H) = 0.0152
Hemorrhage among cesareans: 0.0724 ; among vaginal deliveries: 0.0529


### 11.9 Exercise 7. Reversing a condition

By direct counting, $95/290\approx0.3276$. By reversing the condition, with exact fractions, $P(C\mid H)=\dfrac{(95/1{,}313)(1{,}313/5{,}000)}{290/5{,}000}=\dfrac{95}{290}\approx0.3276$. This is higher than $P(C)=0.2626$: cesareans are more frequent among the hemorrhages than among all deliveries, consistent with the dependence in Exercise 6. Of the 130 infections, 43 followed a vaginal delivery, $43/130\approx0.3308$.

In [20]:
# 1. Among the mothers with hemorrhage, what fraction had a cesarean? Direct count
print("Direct count:", (C & H).sum(), "of", n_hemorrhage, "=", round((C & H).sum() / n_hemorrhage, 4))

# 2. The same by reversing the condition: P(C|H) = P(H|C) P(C) / P(H)
p_h_given_c = (C & H).sum() / n_cesarean
print("Reversal:", round(p_h_given_c, 4), "x", round(n_cesarean / n, 4), "/", round(n_hemorrhage / n, 4), "=", round(p_h_given_c * (n_cesarean / n) / (n_hemorrhage / n), 4))

# 3. Compare with P(C), and look at the vaginal share among the infections
print()
print("P(C) =", round(n_cesarean / n, 4))
print("Vaginal deliveries among the infections:", (~C & I).sum(), "of", n_infection, "=", round((~C & I).sum() / n_infection, 4))

Direct count: 95 of 290 = 0.3276
Reversal: 0.0724 x 0.2626 / 0.058 = 0.3276

P(C) = 0.2626
Vaginal deliveries among the infections: 43 of 130 = 0.3308


### 11.9 Exercise 8. Counting a review of 10 charts

There are $\binom{5{,}000}{10}$ sets of 10 charts, a number with 31 digits, and $10!=3{,}628{,}800$ orders of the chosen charts. The chance that 10 charts include at least one infection is $1-\binom{4{,}870}{10}/\binom{5{,}000}{10}\approx0.2318$. With 27 charts the chance is about 0.5099, and with 26 it is about 0.4968, so 27 is the smallest sample with at least 50%.

In [21]:
# 1. Sets of 10 charts, and orders of the 10 chosen charts
sets_10 = comb(total, 10)
print("Sets of 10 charts:", sets_10, "(", len(str(sets_10)), "digits )")
print("Orders of the 10 chosen charts:", factorial(10))

# 2. Chance that 10 charts include at least one infection (complement)
print("Chance of at least one infection in 10 charts:", round(1 - comb(without_infection, 10) / comb(total, 10), 4))

# 3. Smallest sample with at least a 50% chance
size = 1
while 2 * comb(without_infection, size) > comb(total, size):
    size = size + 1
print("Smallest sample with at least 50%:", size, "charts (", round(1 - comb(without_infection, size - 1) / comb(total, size - 1), 4), "with", size - 1, ")")

Sets of 10 charts: 2667017604016906260066258312000 ( 31 digits )
Orders of the 10 chosen charts: 3628800
Chance of at least one infection in 10 charts: 0.2318
Smallest sample with at least 50%: 27 charts ( 0.4968 with 26 )


### 11.9 Exercise 9. Alternative marginal probabilities

Adding 0.02 and 0.03 gives the union probability only if the overlap probability is zero. The marginals alone do not establish this. The overlap can range from zero to 0.02, so:

$$ 0.03\leq P(H\text{ or }I)\leq0.05. $$

Under the additional assumption of independence:

$$ P(H\text{ or }I)=0.02+0.03-(0.02)(0.03)=0.0494. $$

These are the exercise's alternative probabilities, not the register's proportions.

In [22]:
# 1. Alternative marginals: P(H) = 0.02 and P(I) = 0.03
p_h = 0.02
p_i = 0.03

# 2. Smallest and largest possible value of P(H or I), and its value if H and I were independent
lower = max(p_h, p_i)
upper = min(1, p_h + p_i)
print("P(H or I) must lie between", lower, "and", upper)
print("If the events were independent, P(H or I) =", round(p_h + p_i - p_h * p_i, 4))

P(H or I) must lie between 0.03 and 0.05
If the events were independent, P(H or I) = 0.0494


### 11.9 Exercise 10. Returning charts between draws

With replacement and independent draws, the chance of no infection on each draw remains $4{,}870/5{,}000$. Thus:

$$ \begin{aligned}
P(\text{at least one, with replacement})
 &=1-\left(\frac{4{,}870}{5{,}000}\right)^{20}\approx0.409555,\\
P(\text{at least one, without replacement})
 &=1-\frac{\binom{4{,}870}{20}}{\binom{5{,}000}{20}}\approx0.410156.
\end{aligned} $$

The difference, computed before rounding, is about 0.000600. The original question selects 20 distinct charts without replacement. Both calculations describe selection from existing records, not a comparison of prevention measures.

In [23]:
# 1. Chance of at least one infection in 20 draws, with and without returning the chart
with_replacement = 1 - (without_infection / total) ** 20
without_replacement = 1 - comb(without_infection, 20) / comb(total, 20)
print("Returning each chart:", round(with_replacement, 6), "| not returning:", round(without_replacement, 6))

# 2. The difference between the two procedures
print("The difference is only", round(without_replacement - with_replacement, 6))

Returning each chart: 0.409555 | not returning: 0.410156
The difference is only 0.0006


---
*Synthetic data (seed 11, `generate_data.py`). Teaching values only; nothing here is a clinical recommendation or an estimate for a real unit.*